# RAG-building script

In [ ]:
# 01_build_rag_training_data.py
# RAG-building stage only.
#
# This script:
#   1. Loads and cleans the Kaggle CSV files.
#   2. Keeps only rows with subject/body/answer.
#   3. Splits train/validation.
#   4. Builds the train-only FAISS knowledge base using BAAI/bge-m3.
#   5. Creates RAG-grounded SFT prompt/completion JSONL files.
#
# It does NOT load or fine-tune Llama.
#
# Install:
#   pip install -U pandas numpy scikit-learn tqdm ftfy faiss-cpu FlagEmbedding

import gc
import json
import random
import re
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import faiss
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from FlagEmbedding import BGEM3FlagModel

try:
    import ftfy
except Exception:
    ftfy = None


@dataclass
class Config:
    dataset_dir: str = r"C:\Users\XXXXXXXXXXX\Project_RAG\dataset\Customer IT Support -Ticket Dataset"
    output_dir: str = r"C:\Users\XXXXXXXXXXX\Project_RAG\outputs"

    embedding_model_id: str = "BAAI/bge-m3"

    seed: int = 42
    validation_size: float = 0.10

    # If artifacts already exist, skip rebuilding.
    resume_if_rag_artifacts_exist: bool = True

    # Keep None for full run. Set smaller values for smoke tests.
    max_train_rows: Optional[int] = None
    max_validation_rows: Optional[int] = None

    # Retrieval/RAG settings.
    faiss_candidate_k: int = 16
    sft_context_top_k: int = 3
    embedding_batch_size: int = 16
    embedding_max_length: int = 8192
    max_context_chars_per_ticket: int = 950
    max_query_chars: int = 3500


CFG = Config()

SYSTEM_PROMPT = (
    "You are a multilingual IT customer-support assistant. "
    "Use the retrieved solved tickets as grounding evidence. "
    "Answer in the same language as the customer ticket unless the customer asks otherwise. "
    "Be concise, professional, and action-oriented. "
    "If the retrieved evidence is insufficient, ask for the missing details instead of inventing facts, policies, ETAs, prices, or account-specific information."
)

TEXT_COLUMNS = [
    "subject",
    "body",
    "answer",
    "type",
    "queue",
    "priority",
    "language",
    "business_type",
] + [f"tag_{i}" for i in range(1, 10)]

METADATA_COLUMNS = [
    "type",
    "queue",
    "priority",
    "language",
    "business_type",
] + [f"tag_{i}" for i in range(1, 10)]

MOJIBAKE_MARKERS = ("Ã", "Â", "â€", "â€™", "â€œ", "â€\x9d", "ðŸ")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_dirs(cfg: Config) -> Dict[str, Path]:
    out = Path(cfg.output_dir)

    paths = {
        "output": out,
        "processed": out / "processed",
        "rag_store": out / "rag_store",
        "sft_data": out / "sft_data",
    }

    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)

    return paths


def rag_artifacts_exist(paths: Dict[str, Path]) -> bool:
    required = [
        paths["processed"] / "train.csv",
        paths["processed"] / "validation.csv",
        paths["rag_store"] / "faiss_train.index",
        paths["rag_store"] / "corpus.jsonl",
        paths["rag_store"] / "train_vectors.npy",
        paths["sft_data"] / "train_sft_prompt_completion.jsonl",
        paths["sft_data"] / "validation_sft_prompt_completion.jsonl",
    ]

    return all(path.exists() for path in required)


def repair_text(value: Any) -> str:
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    text = str(value)
    text = text.replace("\\n", "\n")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\ufeff", "").strip()

    if any(marker in text for marker in MOJIBAKE_MARKERS):
        if ftfy is not None:
            text = ftfy.fix_text(text)
        else:
            try:
                repaired = text.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
                if sum(repaired.count(m) for m in MOJIBAKE_MARKERS) < sum(text.count(m) for m in MOJIBAKE_MARKERS):
                    text = repaired
            except Exception:
                pass

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def truncate_text(text: Any, max_chars: int) -> str:
    text = repair_text(text)

    if len(text) <= max_chars:
        return text

    return text[: max_chars - 20].rstrip() + " ...[truncated]"


def read_csv_flexible(path: Path) -> pd.DataFrame:
    errors: List[str] = []

    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
        try:
            df = pd.read_csv(path, encoding=encoding, engine="python", on_bad_lines="skip")
            if df.shape[1] >= 2:
                return df
        except Exception as exc:
            errors.append(f"{encoding}: {exc}")

    raise RuntimeError(f"Could not read CSV file {path}. Tried encodings: {errors}")


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]

    return df


def non_empty(text: Any, min_chars: int = 1) -> bool:
    return len(repair_text(text)) >= min_chars


def extract_tags(row: pd.Series) -> List[str]:
    tags: List[str] = []

    for i in range(1, 10):
        val = repair_text(row.get(f"tag_{i}", ""))
        if val:
            tags.append(val)

    seen = set()
    clean_tags = []

    for tag in tags:
        key = tag.lower()

        if key not in seen:
            clean_tags.append(tag)
            seen.add(key)

    return clean_tags


def ticket_query_text(row: pd.Series, cfg: Config) -> str:
    tags = ", ".join(extract_tags(row))

    parts = [
        f"Subject: {repair_text(row.get('subject', ''))}",
        f"Customer issue: {repair_text(row.get('body', ''))}",
        f"Type: {repair_text(row.get('type', ''))}",
        f"Queue: {repair_text(row.get('queue', ''))}",
        f"Priority: {repair_text(row.get('priority', ''))}",
        f"Language: {repair_text(row.get('language', ''))}",
        f"Business type: {repair_text(row.get('business_type', ''))}",
        f"Tags: {tags}",
    ]

    return truncate_text("\n".join([p for p in parts if not p.endswith(": ")]), cfg.max_query_chars)


def ticket_context_text(row: pd.Series) -> str:
    tags = ", ".join(extract_tags(row))

    parts = [
        f"Ticket ID: {repair_text(row.get('ticket_id', ''))}",
        f"Subject: {repair_text(row.get('subject', ''))}",
        f"Customer issue: {repair_text(row.get('body', ''))}",
        f"Resolved support answer: {repair_text(row.get('answer', ''))}",
        f"Type: {repair_text(row.get('type', ''))}",
        f"Queue: {repair_text(row.get('queue', ''))}",
        f"Priority: {repair_text(row.get('priority', ''))}",
        f"Language: {repair_text(row.get('language', ''))}",
        f"Business type: {repair_text(row.get('business_type', ''))}",
        f"Tags: {tags}",
    ]

    return "\n".join([p for p in parts if not p.endswith(": ")])


def load_answered_tickets(cfg: Config, paths: Dict[str, Path]) -> pd.DataFrame:
    dataset_dir = Path(cfg.dataset_dir)

    if not dataset_dir.exists():
        raise FileNotFoundError(f"Dataset directory does not exist: {dataset_dir}")

    csv_paths = sorted(dataset_dir.glob("*.csv"))

    if not csv_paths:
        raise FileNotFoundError(f"No .csv files found in: {dataset_dir}")

    all_frames: List[pd.DataFrame] = []
    file_summaries: List[Dict[str, Any]] = []

    for csv_path in csv_paths:
        df = normalize_columns(read_csv_flexible(csv_path))
        df["source_file"] = csv_path.name
        df["source_row"] = np.arange(len(df))

        for col in TEXT_COLUMNS:
            if col in df.columns:
                df[col] = df[col].map(repair_text)

        file_summaries.append(
            {
                "file": csv_path.name,
                "rows": int(len(df)),
                "columns": list(df.columns),
                "has_answer_column": bool("answer" in df.columns),
            }
        )

        all_frames.append(df)

    merged = pd.concat(all_frames, ignore_index=True, sort=False)

    for col in ["subject", "body", "answer"] + METADATA_COLUMNS:
        if col not in merged.columns:
            merged[col] = ""

    answered = merged[
        merged["subject"].map(lambda x: non_empty(x, 2))
        & merged["body"].map(lambda x: non_empty(x, 15))
        & merged["answer"].map(lambda x: non_empty(x, 10))
    ].copy()

    answered["language"] = answered["language"].map(lambda x: repair_text(x).lower())
    answered["priority"] = answered["priority"].map(lambda x: repair_text(x).lower())

    dedupe_key = (
        answered["subject"].str.lower().fillna("")
        + "\n"
        + answered["body"].str.lower().fillna("")
        + "\n"
        + answered["answer"].str.lower().fillna("")
    )

    answered = answered.assign(_dedupe_key=dedupe_key)
    answered = answered.drop_duplicates("_dedupe_key").drop(columns=["_dedupe_key"]).reset_index(drop=True)

    answered["ticket_id"] = [f"T{idx:07d}" for idx in range(len(answered))]
    answered["embed_text"] = answered.apply(lambda row: ticket_query_text(row, cfg), axis=1)
    answered["context_text"] = answered.apply(ticket_context_text, axis=1)

    with open(paths["processed"] / "csv_file_summary.json", "w", encoding="utf-8") as f:
        json.dump(file_summaries, f, ensure_ascii=False, indent=2)

    answered.to_csv(paths["processed"] / "combined_answered_tickets.csv", index=False, encoding="utf-8")

    return answered


def split_train_validation(df: pd.DataFrame, cfg: Config, paths: Dict[str, Path]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    stratify = None
    lang_counts = df["language"].value_counts(dropna=False)

    if len(lang_counts) > 1 and (lang_counts >= 2).all():
        stratify = df["language"]

    train_df, val_df = train_test_split(
        df,
        test_size=cfg.validation_size,
        random_state=cfg.seed,
        shuffle=True,
        stratify=stratify,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    if cfg.max_train_rows is not None:
        train_df = train_df.sample(n=min(cfg.max_train_rows, len(train_df)), random_state=cfg.seed).reset_index(drop=True)

    if cfg.max_validation_rows is not None:
        val_df = val_df.sample(n=min(cfg.max_validation_rows, len(val_df)), random_state=cfg.seed).reset_index(drop=True)

    train_df.to_csv(paths["processed"] / "train.csv", index=False, encoding="utf-8")
    val_df.to_csv(paths["processed"] / "validation.csv", index=False, encoding="utf-8")

    return train_df, val_df


def dataframe_to_corpus(df: pd.DataFrame) -> List[Dict[str, Any]]:
    corpus: List[Dict[str, Any]] = []

    for _, row in df.iterrows():
        corpus.append(
            {
                "ticket_id": repair_text(row.get("ticket_id", "")),
                "source_file": repair_text(row.get("source_file", "")),
                "source_row": int(row.get("source_row", -1)) if str(row.get("source_row", "")).strip() else -1,
                "subject": repair_text(row.get("subject", "")),
                "body": repair_text(row.get("body", "")),
                "answer": repair_text(row.get("answer", "")),
                "type": repair_text(row.get("type", "")),
                "queue": repair_text(row.get("queue", "")),
                "priority": repair_text(row.get("priority", "")),
                "language": repair_text(row.get("language", "")),
                "business_type": repair_text(row.get("business_type", "")),
                "tags": extract_tags(row),
                "embed_text": repair_text(row.get("embed_text", "")),
                "context_text": repair_text(row.get("context_text", "")),
            }
        )

    return corpus


def write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def encode_dense(
    model: BGEM3FlagModel,
    texts: List[str],
    cfg: Config,
    desc: str,
) -> np.ndarray:
    vectors: List[np.ndarray] = []

    for start in tqdm(range(0, len(texts), cfg.embedding_batch_size), desc=desc):
        batch = texts[start : start + cfg.embedding_batch_size]
        output = model.encode(batch, batch_size=cfg.embedding_batch_size, max_length=cfg.embedding_max_length)
        dense = np.asarray(output["dense_vecs"], dtype="float32")
        vectors.append(dense)

    arr = np.vstack(vectors).astype("float32")
    faiss.normalize_L2(arr)

    return arr


def build_faiss_index(
    train_df: pd.DataFrame,
    cfg: Config,
    paths: Dict[str, Path],
) -> Tuple[faiss.Index, np.ndarray, List[Dict[str, Any]]]:
    print("Loading BGE-M3 embedding model...")
    embedder = BGEM3FlagModel(cfg.embedding_model_id, use_fp16=torch.cuda.is_available())

    corpus = dataframe_to_corpus(train_df)
    write_jsonl(paths["rag_store"] / "corpus.jsonl", corpus)

    train_texts = [doc["embed_text"] for doc in corpus]

    train_vectors = encode_dense(
        embedder,
        train_texts,
        cfg=cfg,
        desc="Embedding train corpus",
    )

    index = faiss.IndexFlatIP(train_vectors.shape[1])
    index.add(train_vectors)

    faiss.write_index(index, str(paths["rag_store"] / "faiss_train.index"))
    np.save(paths["rag_store"] / "train_vectors.npy", train_vectors)

    del embedder
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return index, train_vectors, corpus


def candidates_for_row(
    ids: np.ndarray,
    scores: np.ndarray,
    corpus: List[Dict[str, Any]],
    exclude_ticket_id: Optional[str] = None,
) -> List[Dict[str, Any]]:
    candidates: List[Dict[str, Any]] = []

    for idx, score in zip(ids.tolist(), scores.tolist()):
        if idx < 0:
            continue

        doc = dict(corpus[int(idx)])

        if exclude_ticket_id and doc["ticket_id"] == exclude_ticket_id:
            continue

        doc["faiss_score"] = float(score)
        candidates.append(doc)

    return candidates


def format_retrieved_context(retrieved_docs: List[Dict[str, Any]], cfg: Config) -> str:
    if not retrieved_docs:
        return "No sufficiently similar solved tickets were retrieved."

    blocks: List[str] = []

    for rank, doc in enumerate(retrieved_docs, start=1):
        tags = ", ".join(doc.get("tags", []))
        score = doc.get("faiss_score", 0.0)

        block = (
            f"[Retrieved ticket {rank}]\n"
            f"Ticket ID: {doc.get('ticket_id', '')}\n"
            f"Relevance score: {score:.4f}\n"
            f"Subject: {truncate_text(doc.get('subject', ''), 250)}\n"
            f"Customer issue: {truncate_text(doc.get('body', ''), cfg.max_context_chars_per_ticket)}\n"
            f"Resolved support answer: {truncate_text(doc.get('answer', ''), cfg.max_context_chars_per_ticket)}\n"
            f"Metadata: type={doc.get('type', '')}; queue={doc.get('queue', '')}; priority={doc.get('priority', '')}; "
            f"language={doc.get('language', '')}; business_type={doc.get('business_type', '')}; tags={tags}"
        )

        blocks.append(block)

    return "\n\n".join(blocks)


def build_user_prompt(row: pd.Series, retrieved_context: str, cfg: Config) -> str:
    tags = ", ".join(extract_tags(row))

    return (
        "Customer ticket to answer:\n"
        f"Subject: {truncate_text(row.get('subject', ''), 300)}\n"
        f"Message:\n{truncate_text(row.get('body', ''), cfg.max_query_chars)}\n\n"
        "Ticket metadata:\n"
        f"type={repair_text(row.get('type', ''))}; queue={repair_text(row.get('queue', ''))}; "
        f"priority={repair_text(row.get('priority', ''))}; language={repair_text(row.get('language', ''))}; "
        f"business_type={repair_text(row.get('business_type', ''))}; tags={tags}\n\n"
        "Retrieved solved-ticket evidence:\n"
        f"{retrieved_context}\n\n"
        "Task: Draft the best customer-support response grounded in the retrieved evidence. "
        "Do not mention internal model behavior. Do not expose private placeholders except when asking the customer to provide missing account/device/log details."
    )


def build_sft_record(row: pd.Series, retrieved_docs: List[Dict[str, Any]], cfg: Config) -> Dict[str, Any]:
    retrieved_context = format_retrieved_context(retrieved_docs, cfg)

    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(row, retrieved_context, cfg)},
        ],
        "completion": [
            {"role": "assistant", "content": repair_text(row.get("answer", ""))}
        ],
    }


def build_sft_dataset_records(
    df: pd.DataFrame,
    ids: np.ndarray,
    scores: np.ndarray,
    corpus: List[Dict[str, Any]],
    cfg: Config,
    desc: str,
    exclude_self: bool,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    sft_records: List[Dict[str, Any]] = []
    audit_records: List[Dict[str, Any]] = []

    for row_idx, row in tqdm(list(df.iterrows()), desc=desc):
        exclude_ticket_id = repair_text(row.get("ticket_id", "")) if exclude_self else None

        candidates = candidates_for_row(
            ids[row_idx],
            scores[row_idx],
            corpus,
            exclude_ticket_id=exclude_ticket_id,
        )

        retrieved = candidates[: cfg.sft_context_top_k]

        sft_record = build_sft_record(row, retrieved, cfg)
        sft_records.append(sft_record)

        audit_records.append(
            {
                "ticket_id": repair_text(row.get("ticket_id", "")),
                "source_file": repair_text(row.get("source_file", "")),
                "language": repair_text(row.get("language", "")),
                "queue": repair_text(row.get("queue", "")),
                "priority": repair_text(row.get("priority", "")),
                "retrieved_ticket_ids": [doc.get("ticket_id", "") for doc in retrieved],
                "retrieved_scores": [doc.get("faiss_score", 0.0) for doc in retrieved],
                "prompt": sft_record["prompt"],
                "completion": sft_record["completion"],
            }
        )

    return sft_records, audit_records


def build_all_sft_records(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    index: faiss.Index,
    train_vectors: np.ndarray,
    corpus: List[Dict[str, Any]],
    cfg: Config,
    paths: Dict[str, Path],
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    print("Searching FAISS for train rows...")
    train_scores, train_ids = index.search(train_vectors.astype("float32"), cfg.faiss_candidate_k)

    print("Embedding validation rows for train-only retrieval...")
    embedder = BGEM3FlagModel(cfg.embedding_model_id, use_fp16=torch.cuda.is_available())

    val_vectors = encode_dense(
        embedder,
        val_df["embed_text"].tolist(),
        cfg=cfg,
        desc="Embedding validation queries",
    )

    val_scores, val_ids = index.search(val_vectors.astype("float32"), cfg.faiss_candidate_k)

    del embedder, val_vectors
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    train_records, train_audit = build_sft_dataset_records(
        train_df,
        train_ids,
        train_scores,
        corpus,
        cfg,
        desc="Building RAG-SFT train examples",
        exclude_self=True,
    )

    val_records, val_audit = build_sft_dataset_records(
        val_df,
        val_ids,
        val_scores,
        corpus,
        cfg,
        desc="Building RAG-SFT validation examples",
        exclude_self=False,
    )

    write_jsonl(paths["sft_data"] / "train_sft_prompt_completion.jsonl", train_records)
    write_jsonl(paths["sft_data"] / "validation_sft_prompt_completion.jsonl", val_records)
    write_jsonl(paths["sft_data"] / "train_sft_audit.jsonl", train_audit)
    write_jsonl(paths["sft_data"] / "validation_sft_audit.jsonl", val_audit)

    return train_records, val_records


def build_rag_sft_from_scratch(cfg: Config, paths: Dict[str, Path]) -> None:
    print("Loading and cleaning dataset...")
    answered = load_answered_tickets(cfg, paths)
    print(f"Usable answered tickets after cleaning/deduplication: {len(answered):,}")

    print("Splitting train/validation...")
    train_df, val_df = split_train_validation(answered, cfg, paths)
    print(f"Train rows: {len(train_df):,} | Validation rows: {len(val_df):,}")

    print("Building train-only FAISS knowledge base...")
    index, train_vectors, corpus = build_faiss_index(train_df, cfg, paths)

    print("Building RAG-grounded SFT records...")
    train_records, val_records = build_all_sft_records(
        train_df=train_df,
        val_df=val_df,
        index=index,
        train_vectors=train_vectors,
        corpus=corpus,
        cfg=cfg,
        paths=paths,
    )

    print(f"SFT train records: {len(train_records):,}")
    print(f"SFT validation records: {len(val_records):,}")

    del index, train_vectors, corpus
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def main() -> None:
    set_seed(CFG.seed)
    paths = ensure_dirs(CFG)

    with open(paths["output"] / "config_rag_building.json", "w", encoding="utf-8") as f:
        json.dump(asdict(CFG), f, ensure_ascii=False, indent=2)

    if CFG.resume_if_rag_artifacts_exist and rag_artifacts_exist(paths):
        print("RAG/SFT artifacts already exist. Skipping RAG rebuilding.")
        print(f"Train split: {paths['processed'] / 'train.csv'}")
        print(f"Validation split: {paths['processed'] / 'validation.csv'}")
        print(f"FAISS index: {paths['rag_store'] / 'faiss_train.index'}")
        print(f"Corpus: {paths['rag_store'] / 'corpus.jsonl'}")
        print(f"Train SFT: {paths['sft_data'] / 'train_sft_prompt_completion.jsonl'}")
        print(f"Validation SFT: {paths['sft_data'] / 'validation_sft_prompt_completion.jsonl'}")
        return

    build_rag_sft_from_scratch(CFG, paths)

    print("\nRAG-building complete.")
    print(f"Processed files saved to: {paths['processed']}")
    print(f"RAG store saved to: {paths['rag_store']}")
    print(f"SFT data saved to: {paths['sft_data']}")


if __name__ == "__main__":
    main()